# Actividad 2: Modelos de Predicción y Clasificación en Machine Learning
* **Repositorio:** https://github.com/andresfkyjasj/ML_2026Q4
* **Dataset:** `scikit-learn/auto-mpg` (Hugging Face Datasets)
* **Dataset URL:** https://huggingface.co/datasets/scikit-learn/auto-mpg

In [ ]:
import pandas as pd
import numpy as np
import datetime
from datasets import load_dataset

# 0. Obtención automática del dataset desde Hugging Face
dataset_id = "scikit-learn/auto-mpg"
print(f"Cargando dataset desde Hugging Face: {dataset_id}...")

hf_dataset = load_dataset(dataset_id, split="train")
df = pd.DataFrame(hf_dataset)

# Estandarización de nombres de columnas
df.columns = [c.strip().lower().replace(' ', '_').replace('-', '_') for c in df.columns]

# Reemplazar '?' por NaN y convertir columnas a numéricas
df = df.replace('?', np.nan)
num_cols = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'mpg']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Muestra del DataFrame e Inspección
print(f"Tamaño del DataFrame: {df.shape[0]} filas x {df.shape[1]} columnas")
print("\nTipos de datos actualizados:")
print(df.dtypes)
print("\nValores nulos reales detectados por columna:")
print(df.isnull().sum())

display(df.head())

Cargando dataset desde Hugging Face: scikit-learn/auto-mpg...
Tamaño del DataFrame: 398 filas x 9 columnas

Columnas estandarizadas: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin', 'car_name']

Tipos de datos:
mpg             float64
cylinders         int64
displacement    float64
horsepower       object
weight            int64
acceleration    float64
model_year        int64
origin            int64
car_name         object
dtype: object

Valores nulos por columna:
mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
car_name        0
dtype: int64


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Definición de columnas
num_features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
cat_features = ['origin']

# Pipeline numérico con imputación por mediana y escalado
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Preprocesador que combina variables numéricas y categóricas
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_features)
])

print("Preprocesador configurado correctamente.")

Preprocesador configurado correctamente.


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Selección de predictoras (se excluye 'mpg' y la columna de texto 'car_name')
cols_to_drop_reg = [c for c in ['mpg', 'car_name'] if c in df.columns]
X_reg = df.drop(columns=cols_to_drop_reg)
y_reg = df['mpg']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Pipeline del modelo de regresión
reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Entrenamiento y Predicción
reg_pipeline.fit(X_train_r, y_train_r)
y_pred_r = reg_pipeline.predict(X_test_r)

# Evaluación
mae_r = mean_absolute_error(y_test_r, y_pred_r)
rmse_r = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
r2_r = r2_score(y_test_r, y_pred_r)

print("--- RESULTADOS REGRESIÓN LINEAL ---")
print(f"MAE:  {mae_r:.4f} mpg")
print(f"RMSE: {rmse_r:.4f} mpg")
print(f"R²:   {r2_r:.4f}")

ValueError: Cannot use median strategy with non-numeric data:
could not convert string to float: '?'

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Creación de variable categórica
def categorizar_eficiencia(mpg_val):
    if mpg_val <= 19.0:
        return 'Baja'
    elif mpg_val <= 29.0:
        return 'Media'
    else:
        return 'Alta'

df['eficiencia_combustible'] = df['mpg'].apply(categorizar_eficiencia)

# Exclusión explícita de 'mpg' y 'car_name' para evitar Target Leakage
cols_to_drop_cls = [c for c in ['mpg', 'car_name', 'eficiencia_combustible'] if c in df.columns]
X_cls = df.drop(columns=cols_to_drop_cls)
y_cls = df['eficiencia_combustible']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Pipeline de Clasificación con Random Forest
cls_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
])

cls_pipeline.fit(X_train_c, y_train_c)
y_pred_c = cls_pipeline.predict(X_test_c)

# Evaluación
acc_c = accuracy_score(y_test_c, y_pred_c)
prec_c = precision_score(y_test_c, y_pred_c, average='weighted')
rec_c = recall_score(y_test_c, y_pred_c, average='weighted')
f1_c = f1_score(y_test_c, y_pred_c, average='weighted')
labels = ['Baja', 'Media', 'Alta']
cm_c = confusion_matrix(y_test_c, y_pred_c, labels=labels)

print("--- RESULTADOS CLASIFICACIÓN (Random Forest) ---")
print(f"Accuracy:  {acc_c:.4f}")
print(f"Precision: {prec_c:.4f}")
print(f"Recall:    {rec_c:.4f}")
print(f"F1-Score:  {f1_c:.4f}")

print("\nMatriz de Confusión:")
print(pd.DataFrame(cm_c, index=[f"Real {l}" for l in labels], columns=[f"Pred {l}" for l in labels]))

In [ ]:
txt_filename = "resumen_modelos.txt"
now_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

txt_content = f"""======================================================================
RESUMEN AUTOMÁTICO DE EJECUCIÓN DE MODELOS DE MACHINE LEARNING
Fecha de ejecución: {now_str}
======================================================================

1. IDENTIFICACIÓN DEL DATASET Y PREPROCESAMIENTO
----------------------------------------------------------------------
- Dataset ID: scikit-learn/auto-mpg (Hugging Face)
- Enlace: https://huggingface.co/datasets/scikit-learn/auto-mpg
- Registros totales: {df.shape[0]} | Columnas: {df.shape[1]}
- Preprocesamiento: Imputación por mediana en 'horsepower', escalado
  estándar en variables continuas y One-Hot Encoding en 'origin'.
- Prevención de Data Leakage: Transformaciones ajustadas solo en Train set.

2. MODELO DE REGRESIÓN LINEAL
----------------------------------------------------------------------
- Objetivo: Estimar la eficiencia continua de combustible en mpg.
- Variable Objetivo: mpg
- Predictoras: cylinders, displacement, horsepower, weight, acceleration, model_year, origin
- Algoritmo: LinearRegression
- Métricas:
  * MAE:  {mae_r:.4f} mpg
  * RMSE: {rmse_r:.4f} mpg
  * R²:   {r2_r:.4f}

3. MODELO DE CLASIFICACIÓN
----------------------------------------------------------------------
- Objetivo: Clasificar categoría de eficiencia (Baja, Media, Alta).
- Variable Objetivo: eficiencia_combustible (Baja <= 19, Media 19-29, Alta > 29)
- Predictoras: Mismas variables que regresión (se excluyó 'mpg' por Data Leakage).
- Algoritmo: RandomForestClassifier (n_estimators=100, max_depth=5)
- Métricas:
  * Accuracy:  {acc_c:.4f}
  * Precision: {prec_c:.4f}
  * Recall:    {rec_c:.4f}
  * F1-Score:  {f1_c:.4f}
- Matriz de Confusión:
{cm_c}

4. CONCLUSIONES
----------------------------------------------------------------------
- Ambos modelos responden eficazmente al problema planteado sobre el dataset Auto-MPG.
======================================================================
"""

with open(txt_filename, "w", encoding="utf-8") as f:
    f.write(txt_content)

print(f"Archivo '{txt_filename}' generado con éxito.")